## learn conditional distribution of median given Medium absolute deviation

In [1]:
using Random, Statistics, Distributions
using Convex, MosekTools
using LinearAlgebra

# ============================================================
# 1) binning in U with merged tails
# ============================================================

"""
Make edges for U with merged tails:
- total bins = nbins_total
- left tail mass ~ q_bounds[1]
- right tail mass ~ 1-q_bounds[2]
- interior bins are quantile-equal between q_bounds[1] and q_bounds[2]
"""
function make_U_edges_merged_tails(U::Vector{Float64};
    nbins_total::Int = 60,
    q_bounds::Tuple{Float64,Float64} = (0.01, 0.99),
    jitter::Float64 = 0.0,
    rng::AbstractRNG = Random.default_rng(),
)
    @assert nbins_total >= 3 "need at least 3 bins (left tail, interior, right tail)"
    qlo, qhi = q_bounds
    @assert 0 < qlo < qhi < 1

    Uuse = jitter > 0 ? (U .+ jitter .* randn(rng, length(U))) : U

    interior_bins = nbins_total - 2
    probs = collect(range(qlo, qhi; length=interior_bins+1))   # includes qlo, qhi
    qs = quantile(Uuse, probs)                                 # length interior_bins+1
    cuts = qs[2:end-1]                                         # exclude endpoints

    # enforce strictly increasing cuts (handle ties)
    cuts = unique(cuts)
    if length(cuts) < interior_bins-1
        @warn "Many tied quantiles: interior bins reduced from $interior_bins to $(length(cuts)+1)"
    end

    edges = vcat(-Inf, cuts, Inf)
    return edges
end

"""
Assign each U[i] to a bin in 1:nbins (edges length = nbins+1).
"""
function bin_ids(U::Vector{Float64}, edges::Vector{Float64})
    nbins = length(edges) - 1
    b = searchsortedlast.(Ref(edges), U)
    return clamp.(b, 1, nbins)
end

function counts_by_edges(U::Vector{Float64}, edges::Vector{Float64})
    nbins = length(edges)-1
    b = searchsortedlast.(Ref(edges), U)
    b = clamp.(b, 1, nbins)
    counts = zeros(Int, nbins)
    @inbounds for bi in b
        counts[bi] += 1
    end
    return counts
end

# ============================================================
# 2) beta grid for mixture components
# ============================================================

"""
Default grid for mixture on beta.
"""
function default_grid_for_beta(β::Vector{Float64}; n_mu=25, n_sigma=40)
    qlo, qhi = quantile(β, (0.001, 0.999))
    pad = 0.2 * (qhi - qlo)
    mus = collect(range(qlo - pad, qhi + pad; length=n_mu))

    sβ = std(β)
    σ_lo = max(1e-3, 0.05 * sβ)
    σ_hi = max(σ_lo * 2, 2.0 * sβ)
    sigmas = exp.(range(log(σ_lo), log(σ_hi); length=n_sigma))

    return mus, sigmas
end

# ============================================================
# 6) h_hat: compute P(|β| >= t | U-bin) using fitted mixture
# ============================================================

@inline function two_sided_tail(d::UnivariateDistribution, t::Float64)
    tt = abs(t)
    return ccdf(d, tt) + cdf(d, -tt)
end

"""
Given fitted per-bin conditional mixture, compute h(t,v)
where v is D (not log D).
"""
function h_hat(fit, t::Float64, v::Float64)
    u = log(v + 1e-12)
    edges = fit.edges
    nbins = length(edges) - 1
    b = clamp(searchsortedlast(edges, u), 1, nbins)

    π = fit.π_bins[b]
    isempty(π) && return NaN

    meta = fit.comps
    s = 0.0
    @inbounds for k in 1:length(π)
        dk = component_dist(meta, k)
        s += π[k] * two_sided_tail(dk, t)
    end
    return s
end

h_hat

In [2]:

# ============================================================
# 3) mixed grid: Normal + Student-t(ν) components
# ============================================================


function build_mixed_grid_separate(
    musN::Vector{Float64}, sigmasN::Vector{Float64};
    musT::Vector{Float64}=musN, sigmasT::Vector{Float64}=sigmasN,
    dofs::Vector{Int}=Int[3,5,8],
    include_normals::Bool=true,
    include_t::Bool=true
)
    mu_list   = Float64[]
    sig_list  = Float64[]
    fam_list  = Symbol[]
    dof_list  = Int[]

    if include_normals
        for μ in musN, σ in sigmasN
            push!(mu_list, μ); push!(sig_list, σ)
            push!(fam_list, :normal); push!(dof_list, 0)
        end
    end

    if include_t
        for ν in dofs, μ in musT, σ in sigmasT
            push!(mu_list, μ); push!(sig_list, σ)
            push!(fam_list, :t); push!(dof_list, ν)
        end
    end

    return (mu=mu_list, sigma=sig_list, family=fam_list, dof=dof_list)
end

@inline function component_dist(meta, k::Int)
    μ = meta.mu[k]
    σ = meta.sigma[k]
    if meta.family[k] === :normal
        return Normal(μ, σ)
    else
        ν = meta.dof[k]
        return LocationScale(μ, σ, TDist(ν))
    end
end

# ============================================================
# 4) fit mixture weights with MOSEK/Convex on fixed grid
# ============================================================

# """
# Fit weights π on a fixed grid of components (Normal + t).
# Objective: maximize Σ_i w_i log( Σ_k π_k f_k(x_i) )
# using log-sum-exp stabilization.

function fit_fixedgrid_mixed_mixture_mosek_weighted(
    x::Vector{Float64},
    w::Vector{Float64};
    musN::Vector{Float64},
    sigmasN::Vector{Float64},
    musT::Vector{Float64}=musN,
    sigmasT::Vector{Float64}=sigmasN,
    dofs::Vector{Int} = Int[3,5,8],
    include_normals::Bool = true,
    include_t::Bool = true,
    verbose::Bool=false
)
    @assert length(x) == length(w)
    N = length(x)

    meta = build_mixed_grid_separate(
        musN, sigmasN;
        musT=musT, sigmasT=sigmasT,
        dofs=dofs,
        include_normals=include_normals,
        include_t=include_t
    )
    K = length(meta.mu)

    logφ = Matrix{Float64}(undef, N, K)
    @inbounds for k in 1:K
        dk = component_dist(meta, k)
        for i in 1:N
            logφ[i, k] = logpdf(dk, x[i])
        end
    end

    m = vec(maximum(logφ, dims=2))
    A = exp.(clamp.(logφ .- m, -745.0, 0.0))

    π = Variable(K)
    constraints = [π >= 0, sum(π) == 1]
    obj = dot(w, m) + sum(w .* log(A * π))

    problem = maximize(obj, constraints)
    solve!(problem, Mosek.Optimizer; silent=!verbose)

    π_hat = vec(evaluate(π))
    π_hat = max.(π_hat, 0.0)
    π_hat ./= sum(π_hat)

    return π_hat, meta
end

fit_fixedgrid_mixed_mixture_mosek_weighted (generic function with 1 method)

In [3]:
# ============================================================
# 2b) within-bin β-based keep probabilities / weights
# ============================================================

"""
Piecewise keep probability based on z = abs.(βb) within a bin.

- z >= q_beta_tail quantile  -> keep prob = 1
- z <  q_beta_tail quantile  -> keep prob = a_min_beta

Returns:
  aβ::Vector{Float64}, info::NamedTuple
"""
function beta_keep_prob_piecewise(
    βb::Vector{Float64};
    q_beta_tail::Float64 = 0.95,
    a_min_beta::Float64 = 0.10
)
    @assert 0 < q_beta_tail < 1
    @assert 0 < a_min_beta <= 1

    z = abs.(βb)
    zR = quantile(z, q_beta_tail)

    aβ = similar(z, Float64)
    @inbounds for i in eachindex(z)
        aβ[i] = (z[i] >= zR) ? 1.0 : a_min_beta
    end

    info = (
        rule = :piecewise,
        q_beta_tail = q_beta_tail,
        zR = zR,
        a_min_beta = a_min_beta,
        frac_tail = mean(z .>= zR),
        mean_keep_prob = mean(aβ)
    )
    return aβ, info
end


"""
Apply β-based weighting or subsampling inside a bin.

Modes:
- beta_keep_rule = :none       -> returns original βb and all-ones weights
- beta_keep_rule = :piecewise  -> uses beta_keep_prob_piecewise

If beta_use_subsample=true:
  randomly keep points with prob aβ and assign IPW weights = 1/aβ on kept points
Else:
  keep all points and assign weights proportional to 1/aβ (reweight-only)

Returns:
  β_use, w_use, info
"""
function prepare_within_bin_beta_weighting(
    βb::Vector{Float64};
    rng::AbstractRNG = Random.default_rng(),
    beta_keep_rule::Symbol = :none,
    q_beta_tail::Float64 = 0.95,
    a_min_beta::Float64 = 0.10,
    beta_use_subsample::Bool = false,
    rescale_beta_weights::Bool = true
)
    n = length(βb)
    n == 0 && return βb, Float64[], (rule=:none, n_in=0, n_used=0)

    # no weighting
    if beta_keep_rule == :none
        return βb, ones(Float64, n), (rule=:none, n_in=n, n_used=n, mean_weight=1.0, max_weight=1.0)
    end

    # get keep probs
    if beta_keep_rule == :piecewise
        aβ, info0 = beta_keep_prob_piecewise(βb; q_beta_tail=q_beta_tail, a_min_beta=a_min_beta)
    else
        error("Unknown beta_keep_rule=$beta_keep_rule. Supported: :none, :piecewise")
    end

    if beta_use_subsample
        keep = rand(rng, n) .< aβ
        idx = findall(keep)

        β_use = βb[idx]
        w_use = 1.0 ./ aβ[idx]

        if rescale_beta_weights && !isempty(w_use)
            # normalize so sum(weights) ≈ number used
            w_use .*= (length(w_use) / sum(w_use))
        end

        info = (; info0..., n_in=n, n_used=length(β_use),
                 used_frac=length(β_use)/n,
                 mean_weight=(isempty(w_use) ? NaN : mean(w_use)),
                 max_weight=(isempty(w_use) ? NaN : maximum(w_use)))
        return β_use, w_use, info
    else
        # weight-only, no subsampling
        β_use = βb
        w_use = 1.0 ./ aβ

        if rescale_beta_weights && !isempty(w_use)
            w_use .*= (length(w_use) / sum(w_use))
        end

        info = (; info0..., n_in=n, n_used=n, used_frac=1.0,
                 mean_weight=mean(w_use), max_weight=maximum(w_use))
        return β_use, w_use, info
    end
end

prepare_within_bin_beta_weighting

In [4]:
function fit_conditional_beta_given_U_bins(
    β::Vector{Float64},
    U::Vector{Float64};
    nbins_total::Int = 60,
    q_bounds::Tuple{Float64,Float64} = (0.01,0.99),
    jitter::Float64 = 1e-10,
    rng::AbstractRNG = Random.default_rng(),

    # separate grid sizes
    n_mu_N::Int = 25,
    n_sigma_N::Int = 40,
    n_mu_T::Int = 9,
    n_sigma_T::Int = 15,

    dofs::Vector{Int} = Int[3,5,8,12,20],
    include_normals::Bool = true,
    include_t::Bool = true,
    min_bin_size::Int = 500,
    verbose::Bool = false,
    bins_to_fit::Union{Nothing, Vector{Int}} = nothing,

    # optional: widen t sigmas upward for tail focus
    widen_t_sigma_mult::Float64 = 1.0,

    # NEW: within-bin β-based weighting/subsampling
    beta_keep_rule::Symbol = :none,       # :none or :piecewise
    q_beta_tail::Float64 = 0.95,          # top 5% |β| treated as tail
    a_min_beta::Float64 = 0.10,           # center keep prob
    beta_use_subsample::Bool = false,     # false = weight-only; true = actual subsample+IPW
    rescale_beta_weights::Bool = true
)
    @assert length(β) == length(U)

    edges = make_U_edges_merged_tails(U; nbins_total=nbins_total,
                                      q_bounds=q_bounds, jitter=jitter, rng=rng)

    nbins = length(edges) - 1
    b_id = bin_ids(U, edges)

    # shared grids across bins
    musN, sigmasN = default_grid_for_beta(β; n_mu=n_mu_N, n_sigma=n_sigma_N)
    musT, sigmasT = default_grid_for_beta(β; n_mu=n_mu_T, n_sigma=n_sigma_T)

    if widen_t_sigma_mult != 1.0
        sigmasT = exp.(range(log(minimum(sigmasT)),
                             log(maximum(sigmasT) * widen_t_sigma_mult);
                             length=length(sigmasT)))
    end

    π_bins = [Float64[] for _ in 1:nbins]
    counts = zeros(Int, nbins)              # original count in each bin
    counts_used = zeros(Int, nbins)         # actual rows used in fit (after β subsampling if enabled)
    weight_sums = zeros(Float64, nbins)     # sum of weights used in fit
    comps_ref = nothing

    # store diagnostics per bin
    # beta_weight_stats = [nothing for _ in 1:nbins]
    beta_weight_stats = Vector{Any}(undef, nbins)
    fill!(beta_weight_stats, nothing)

    fit_set = bins_to_fit === nothing ? collect(1:nbins) : unique(clamp.(bins_to_fit, 1, nbins))
    fit_flag = falses(nbins)
    fit_flag[fit_set] .= true

    for b in 1:nbins
        idx = findall(==(b), b_id)
        counts[b] = length(idx)

        if !fit_flag[b] || counts[b] < min_bin_size
            continue
        end

        βb = β[idx]

        # NEW: within-bin β-based weighting / subsampling
        β_use, wb, infoβ = prepare_within_bin_beta_weighting(
            βb;
            rng=rng,
            beta_keep_rule=beta_keep_rule,
            q_beta_tail=q_beta_tail,
            a_min_beta=a_min_beta,
            beta_use_subsample=beta_use_subsample,
            rescale_beta_weights=rescale_beta_weights
        )

        beta_weight_stats[b] = infoβ
        counts_used[b] = length(β_use)
        weight_sums[b] = isempty(wb) ? 0.0 : sum(wb)

        # if after subsampling too few points remain, skip
        if counts_used[b] < min_bin_size
            continue
        end

        π_hat, meta = fit_fixedgrid_mixed_mixture_mosek_weighted(
            β_use, wb;
            musN=musN, sigmasN=sigmasN,
            musT=musT, sigmasT=sigmasT,
            dofs=dofs,
            include_normals=include_normals,
            include_t=include_t,
            verbose=verbose
        )

        π_bins[b] = π_hat
        comps_ref === nothing && (comps_ref = meta)
    end

    return (
        edges=edges, π_bins=π_bins, comps=comps_ref,
        counts=counts, counts_used=counts_used, weight_sums=weight_sums,
        bins_fit=fit_set,
        grids=(musN=musN, sigmasN=sigmasN, musT=musT, sigmasT=sigmasT),
        beta_weight_rule=beta_keep_rule,
        beta_use_subsample=beta_use_subsample,
        beta_weight_stats=beta_weight_stats
    )
end

fit_conditional_beta_given_U_bins (generic function with 1 method)

In [6]:
# median absolute deviation about the median
@inline mad_med(x) = median(abs.(x .- median(x)))

"""
Simulate B datasets of size n from N(0,1), return:
  beta_tilde = sqrt(n) * median(x)
  D = median(|x - median(x)|)     # MAD about median
  U = log(D)
"""
function simulate_beta_Dmed_U(n::Int, B::Int; rng=Random.default_rng())
    β = Vector{Float64}(undef, B)
    D = Vector{Float64}(undef, B)
    x = Vector{Float64}(undef, n)
    dist = Normal(0,1)

    @inbounds for b in 1:B
        rand!(rng, dist, x)
        mmed = median(x)
        dmed = median(abs.(x .- mmed))
        β[b] = sqrt(n) * mmed
        D[b] = dmed
    end

    U = log.(D .+ 1e-300)
    return β, D, U
end

simulate_beta_Dmed_U

## n=3

In [7]:
rng = MersenneTwister(3)
n = 3
B = 1100_000_000
nbins_total = 102
β, D, U = simulate_beta_Dmed_U(n, B; rng=rng)
fit_beta_subsample = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    # bins_to_fit=[1, 10, 20, 70, 80],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,       # <-- actual thinning
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 67.33 seconds, 66.704 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240231 (720693 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241341          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.41            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 73.29 seconds, 59.525 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218813 (656439 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219923          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 81.65 seconds, 59.380 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218509 (655527 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219619          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.65            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 81.09 seconds, 59.472 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218605 (655815 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219715          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.55            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524461 bytes.

(edges = [-Inf, -5.034526459330268, -4.385290035946882, -3.992848351740317, -3.7104371351379215, -3.4890915852149003, -3.3068894451417097, -3.1519729631842206, -3.017134450640217, -2.897541283114484  …  0.010737809018913387, 0.04671866185257391, 0.08521375827129739, 0.12688691240075964, 0.17282161724445902, 0.22470326511098324, 0.2854825305045093, 0.36133738489053263, 0.4696417507178694, Inf], π_bins = [[0.0, 0.0, 2.3030911716742708e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 1.8181835983895003e-6, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 7.13798025668125e-8, 9.488261399014301e-6  …  0.0, 0.0, 0.0, 0.0, 5.42622251761449e-9, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 4.678300707882906e-8, 3.3147179842502724e-8, 3.856463870014875e-6, 0.0, 7.579610724248665e-8  …  0.0, 0.0, 0.0, 0.0, 0.0, 3.096388814376758e-6, 1.0301565976427384e-7, 0.0, 0.0, 0.0], [2.1613044951201722e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 9.787496958798759e-9, 0.0, 2.423991

In [8]:
using JLD2
@save "fit_beta_subsample_1.2_median(n=3).jld2" fit_beta_subsample

## n=4

In [6]:
rng = MersenneTwister(3)
n = 4
B = 1100_000_000
nbins_total = 102
β, D, U = simulate_beta_Dmed_U(n, B; rng=rng)
fit_beta_subsample = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    # bins_to_fit=[1, 10, 20, 70, 80],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,       # <-- actual thinning
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 65.13 seconds, 66.887 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240946 (722838 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 242056          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.41            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 72.3 seconds, 59.348 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218125 (654375 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219235          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.41            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 75.9 seconds, 59.466 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218168 (654504 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219278          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.60            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 79.88 seconds, 59.376 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218233 (654699 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219343          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.57            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524370 bytes.

(edges = [-Inf, -2.972899791128835, -2.639195189828708, -2.4359110171113505, -2.2884837418295136, -2.17239692705253, -2.0762374825615084, -1.9940855232046446, -1.9221738979874967, -1.8581244772560974  …  -0.10078867568165475, -0.07411071545572746, -0.04533938802288597, -0.013891478569259619, 0.021099852656751624, 0.061043351111232824, 0.1084521376980512, 0.16854430244660107, 0.2560931217222825, Inf], π_bins = [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.380632733421113e-9, 0.0  …  0.0, 0.0, 0.0, 0.0, 3.3522049529777086e-6, 0.0, 1.7480257521059995e-7, 0.0, 0.0, 0.0], [0.0, 3.79131104605754e-9, 0.0, 0.0, 0.0, 9.504256912118577e-7, 1.4744197808776834e-7, 1.863151465576804e-8, 0.0, 0.0  …  0.0, 0.0, 1.2904925917043467e-6, 0.0, 0.0, 4.815229396104746e-8, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.811462630812095e-6, 2.7990619391767353e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0

In [ ]:
using JLD2
@save "fit_beta_subsample_1.2_median(n=4).jld2" fit_beta_subsample

In [9]:
using JLD2
@save "fit_beta_subsample_1.2(n=4).jld2" fit_beta_subsample

## n=5

In [7]:
rng = MersenneTwister(3)
n = 5
B = 1100_000_000
nbins_total = 102
β, D, U = simulate_beta_Dmed_U(n, B; rng=rng)
fit_beta_subsample = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    # bins_to_fit=[1, 10, 20, 70, 80],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,       # <-- actual thinning
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 67.75 seconds, 66.698 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240208 (720624 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241318          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.40            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 76.65 seconds, 59.453 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218532 (655596 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219642          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.41            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 78.85 seconds, 59.416 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218303 (654909 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219413          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.60            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 75.38 seconds, 59.485 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218658 (655974 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219768          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.59            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524311 bytes.

(edges = [-Inf, -2.9064530075540103, -2.569872249796174, -2.3645160147856843, -2.215338598884279, -2.0977298569443663, -2.0002675687428315, -1.9167974620584627, -1.8437016938066173, -1.778620176886352  …  0.022405142884556715, 0.04963539429785759, 0.07898826507780704, 0.1110207557550143, 0.14666494675454603, 0.18728305484984378, 0.23546762730766577, 0.296468396050307, 0.38530219529563264, Inf], π_bins = [[1.8752867367525172e-6, 9.261428169130864e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.7866686669167678e-8, 0.0, 0.0, 0.0], [0.0, 0.0, 4.300060835307569e-8, 1.3164706995352625e-6, 2.4605316674921026e-8, 0.0, 0.0, 0.0, 5.3033479266555104e-6, 1.0711176954180444e-6  …  0.0, 0.0, 0.0, 1.903318606357949e-8, 0.0, 0.0, 1.914957077926182e-7, 0.0, 0.0, 0.0], [2.4059713693588723e-6, 2.2944347601778486e-8, 3.6784515343256333e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.8725546578125347e-8  …  0.0, 0.0, 0.0, 5.159085912159981e-6, 0.0, 2.796112375336422e-7, 1.25980994494364

In [8]:
using JLD2
@save "fit_beta_subsample_1.2_median(n=5).jld2" fit_beta_subsample

## n=6

In [7]:
rng = MersenneTwister(3)
n = 6
B = 1100_000_000
nbins_total = 102
# bins10 = choose_bins_evenly(nbins_total, 2)
β, D, U = simulate_beta_Dmed_U(n, B; rng=rng)
# increase bB = 171_000_000, a_min_beta=0.01
fit_beta_subsample = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    # bins_to_fit=[1, 10, 20, 70, 80],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,       # <-- actual thinning
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 71.79 seconds, 66.688 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240170 (720510 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241280          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.52            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 73.65 seconds, 59.459 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218554 (655662 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219664          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.44            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 76.76 seconds, 59.439 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218069 (654207 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219179          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.57            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 79.59 seconds, 59.470 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218600 (655800 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219710          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.54            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524294 bytes.

(edges = [-Inf, -2.2368427649643254, -2.0028774693122897, -1.8588679705888282, -1.7536599717397356, -1.6702779710180031, -1.6009586780597709, -1.541406009674544, -1.489050596276665, -1.4422459964905803  …  -0.05106490926878196, -0.02785394106132789, -0.0027338095600668725, 0.024813653761360188, 0.05560064836924053, 0.09090883147700332, 0.13301867512691626, 0.18673786475211493, 0.26578256291198027, Inf], π_bins = [[0.0, 1.998752975142353e-8, 1.6307444670597904e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 2.0737285284400987e-7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.5187603725706582e-7, 2.4262963761304916e-8, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.971811458585572e-8, 0.0, 0.0, 0.0], [6.3539593593857236e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3546315226723124e-8, 0.0  …  0.0, 0.0, 0.0, 4.2849405813723154e-8, 0.0, 0.0, 0.0, 1.2080916174930772e-7, 0.0, 0.0], [1.0623967442126635e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 1.292602

In [8]:
using JLD2
@save "fit_beta_subsample_1.2_median(n=6).jld2" fit_beta_subsample

In [9]:
fit_beta_subsample

(edges = [-Inf, -2.2368427649643254, -2.0028774693122897, -1.8588679705888282, -1.7536599717397356, -1.6702779710180031, -1.6009586780597709, -1.541406009674544, -1.489050596276665, -1.4422459964905803  …  -0.05106490926878196, -0.02785394106132789, -0.0027338095600668725, 0.024813653761360188, 0.05560064836924053, 0.09090883147700332, 0.13301867512691626, 0.18673786475211493, 0.26578256291198027, Inf], π_bins = [[0.0, 1.998752975142353e-8, 1.6307444670597904e-8, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 2.0737285284400987e-7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.5187603725706582e-7, 2.4262963761304916e-8, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.971811458585572e-8, 0.0, 0.0, 0.0], [6.3539593593857236e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.3546315226723124e-8, 0.0  …  0.0, 0.0, 0.0, 4.2849405813723154e-8, 0.0, 0.0, 0.0, 1.2080916174930772e-7, 0.0, 0.0], [1.0623967442126635e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 1.292602

## n=8

In [6]:
rng = MersenneTwister(3)
n = 8
B = 1100_000_000
nbins_total = 102
# bins10 = choose_bins_evenly(nbins_total, 2)
β, D, U = simulate_beta_Dmed_U(n, B; rng=rng)
# increase bB = 171_000_000, a_min_beta=0.01
fit_beta_subsample = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    # bins_to_fit=[1, 10, 20, 70, 80],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,       # <-- actual thinning
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 66.87 seconds, 66.668 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240091 (720273 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241201          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 72.22 seconds, 59.558 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218940 (656820 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 220050          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.42            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 84.74 seconds, 59.440 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218083 (654249 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219193          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.51            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 76.99 seconds, 59.420 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218404 (655212 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219514          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.56            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524370 bytes.

(edges = [-Inf, -1.8715298967502536, -1.6864572189703153, -1.5719126537362889, -1.4878002752327115, -1.4208750840526791, -1.3650100275878683, -1.3168704539619782, -1.2744282563166196, -1.2363941694227143  …  -0.05463343310613815, -0.03395847987973005, -0.011559491989106802, 0.013047992063202053, 0.04059766561782385, 0.07226024050619992, 0.11012707231163266, 0.15853172065399473, 0.2300811950803782, Inf], π_bins = [[0.0, 0.0, 6.479365476235635e-9, 0.0, 0.0, 0.0, 0.0, 3.6966027252215694e-6, 3.9775073893790873e-7, 1.0558334910331072e-7  …  0.0, 0.0, 8.852426249327287e-9, 0.0, 1.0246471763677918e-8, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0336466267732072e-8, 0.0, 1.6034084326033315e-8, 4.566427288752717e-9, 0.0, 1.4970209767789427e-7  …  0.0, 0.0, 7.620609156720853e-8, 6.739651434172798e-7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.6715564169475718e-8, 0.0, 4.0777143483266774e-9, 0.0, 0.0, 3.3980607414946765e-6  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.

In [9]:
fit_beta_subsample

(edges = [-Inf, -1.8715298967502536, -1.6864572189703153, -1.5719126537362889, -1.4878002752327115, -1.4208750840526791, -1.3650100275878683, -1.3168704539619782, -1.2744282563166196, -1.2363941694227143  …  -0.05463343310613815, -0.03395847987973005, -0.011559491989106802, 0.013047992063202053, 0.04059766561782385, 0.07226024050619992, 0.11012707231163266, 0.15853172065399473, 0.2300811950803782, Inf], π_bins = [[0.0, 0.0, 6.479365476235635e-9, 0.0, 0.0, 0.0, 0.0, 3.6966027252215694e-6, 3.9775073893790873e-7, 1.0558334910331072e-7  …  0.0, 0.0, 8.852426249327287e-9, 0.0, 1.0246471763677918e-8, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0336466267732072e-8, 0.0, 1.6034084326033315e-8, 4.566427288752717e-9, 0.0, 1.4970209767789427e-7  …  0.0, 0.0, 7.620609156720853e-8, 6.739651434172798e-7, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.6715564169475718e-8, 0.0, 4.0777143483266774e-9, 0.0, 0.0, 3.3980607414946765e-6  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.

In [10]:
using JLD2
@save "fit_beta_subsample_1.2_median(n=8).jld2" fit_beta_subsample

## n=10

In [7]:
rng = MersenneTwister(3)
n = 10
B = 1100_000_000
nbins_total = 102
# bins10 = choose_bins_evenly(nbins_total, 2)
β, D, U = simulate_beta_Dmed_U(n, B; rng=rng)
# increase bB = 171_000_000, a_min_beta=0.01
fit_beta_subsample = fit_conditional_beta_given_U_bins(
    β, U;
    nbins_total=102,
    q_bounds=(0.001, 0.999),
    n_mu_N=25, n_sigma_N=30,
    n_mu_T=10, n_sigma_T=12,
    dofs=Int[200, 250, 300],
    # bins_to_fit=[1, 10, 20, 70, 80],
    verbose=true,

    beta_keep_rule=:piecewise,
    q_beta_tail=0.99,
    a_min_beta=0.01,
    beta_use_subsample=true,       # <-- actual thinning
    rescale_beta_weights=true
)

[ Info: [Convex.jl] Compilation finished: 65.98 seconds, 66.789 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 240561 (721683 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 241671          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.43            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 73.34 seconds, 59.526 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218817 (656451 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219927          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.47            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 76.99 seconds, 59.437 GiB of memory allocated


...

Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218461 (655383 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219571          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.63            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

┌ Warning: Problem wasn't solved optimally
│   status = SLOW_PROGRESS::TerminationStatusCode = 19
└ @ Convex ~/.julia/packages/Convex/b2kc4/src/solution.jl:124
[ Info: [Convex.jl] Compilation finished: 85.62 seconds, 59.471 GiB of memory allocated


Problem
  Name                   :                 
  Objective sense        : maximize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 1111            
  Affine conic cons.     : 218601 (655803 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 219711          
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.56            
Lin. dep.  - primal attempts        : 1                 successes              : 1               
Lin. dep.  - dual attempts          : 0       

Excessive output truncated after 524383 bytes.

(edges = [-Inf, -1.6490754765088294, -1.4935621731483562, -1.3967969696149294, -1.3255074505452589, -1.2686324022037374, -1.2210395042466484, -1.1799333375973768, -1.1436487565731452, -1.1110378645124364  …  -0.06746127992067522, -0.04860056947525575, -0.028160365450084508, -0.005657321369834549, 0.019570171501566414, 0.048606903334016474, 0.08339368814458828, 0.12803581404882913, 0.19416781123221574, Inf], π_bins = [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.1644098695004195e-7, 5.082334060969327e-6  …  0.0, 0.0, 0.0, 0.0, 4.0272426613675326e-8, 1.4014699440457443e-7, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0020254500123511e-6, 9.808625646433712e-8, 0.0, 1.4781006599402364e-8, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 1.972810083796026e-7, 0.0, 1.1041153831372666e-7, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5935224565729038e-9, 6.752943315501906e-8, 0.0  …  5.457967285686947e-10, 0.0, 0.0, 6.790354640653632e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

In [8]:
using JLD2
@save "fit_beta_subsample_1.2_median(n=10).jld2" fit_beta_subsample